In [ ]:
import pandas as pd
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DATA_DIR = Path("../data/processed")
df = pd.read_csv(DATA_DIR / "tcga_brca_survival_clean.csv")

three_years = 36.0

ml_df = df.copy()

ml_df["death_within_3y"] = None

ml_df.loc[
    (ml_df["os_event"] == 1) & (ml_df["os_time_months"] <= three_years),
    "death_within_3y"
] = 1

ml_df.loc[
    ml_df["os_time_months"] > three_years,
    "death_within_3y"
] = 0

ml_df = ml_df.dropna(subset=["death_within_3y", "age", "stage_group"])

ml_df["death_within_3y"] = ml_df["death_within_3y"].astype(int)

X = ml_df[["age", "stage_group"]]
y = ml_df["death_within_3y"]

numeric_features = ["age"]
categorical_features = ["stage_group"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(drop="first"), categorical_features),
    ]
)

model = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000)),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

model.fit(X_train, y_train)

pred_prob = model.predict_proba(X_test)[:, 1]
pred_label = model.predict(X_test)

auc = roc_auc_score(y_test, pred_prob)

print("AUC:", round(auc, 3))
print(classification_report(y_test, pred_label))